### Links and Resources
| Concept / Command | Description |
|---|---|
| [`StreamingQuery`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.streaming.StreamingQuery.html) | The handle returned by `.start()` — status, progress, stop |
| [`StreamingQueryListener`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.streaming.StreamingQueryListener.html) | Event-driven callbacks: started, progress, terminated |
| [`spark.streams`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.streaming.StreamingQueryManager.html) | Manage all active queries in the session |
| [Monitoring docs](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#monitoring-streaming-queries) | Full monitoring reference |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

DATA_ROOT = "/Volumes/population_metrics/landing/datasets"

# Monitoring and Production Patterns

A streaming query that silently stops processing is worse than one that fails loudly. This notebook covers the tools and patterns to keep streaming pipelines observable, recoverable, and cost-efficient.

Topics:
1. `query.status` — is it alive?
2. `query.lastProgress` — what happened in the last batch?
3. `query.recentProgress` — throughput trends
4. `StreamingQueryListener` — event-driven monitoring
5. `spark.streams` — managing all active queries
6. `awaitTermination` patterns — clean shutdown and error propagation
7. Common production pitfalls — causes, symptoms, fixes
8. Production checklist

## 1. `query.status` — is it alive?

`query.status` is a lightweight dict available at any time without waiting for a batch to complete.

```python
{
  "message":          "Waiting for data to arrive",   # or "Processing new data"
  "isDataAvailable":  False,
  "isTriggerActive":  False
}
```

| Field | Meaning |
|---|---|
| `message` | Human-readable phase description |
| `isDataAvailable` | New input data exists (but may not have started processing) |
| `isTriggerActive` | A micro-batch is currently executing |

In [ ]:
import time, uuid
from pyspark.sql import functions as F

CKPT_BASE = f"{DATA_ROOT}/streaming_demo/_checkpoints"

# rate-micro-batch (not plain rate) so availableNow can terminate the stream — see notebook 02.
# Under availableNow these monitoring demos see one batch instead of an indefinite stream;
# the API calls (status / lastProgress / recentProgress) work the same.
events = (spark.readStream
    .format("rate-micro-batch").option("rowsPerBatch", 30).option("numPartitions", 1).load()
    .withColumnRenamed("timestamp", "event_time")
    .withColumnRenamed("value",     "event_id")
    .withColumn("user_id", (F.col("event_id") % 5).cast("int"))
    .withColumn("amount",  (F.rand() * 100 + 1).cast("int")))

q = (events.writeStream
     .format("memory").queryName("monitor_demo")
     .outputMode("append")
     .option("checkpointLocation", f"{CKPT_BASE}/monitor_demo_{uuid.uuid4().hex[:8]}")
     .trigger(availableNow=True)
     .start())

q.awaitTermination()
s = q.status
print(f"isActive:         {q.isActive}")
print(f"message:          {s['message']}")
print(f"isDataAvailable:  {s['isDataAvailable']}")
print(f"isTriggerActive:  {s['isTriggerActive']}")
print(f"id:               {q.id}")
print(f"runId:            {q.runId}  (new runId each time the query starts)")

## 2. `query.lastProgress` — what happened in the last batch?

`lastProgress` is populated after every completed batch. It contains the richest diagnostics available at the query level.

In [ ]:
# Under availableNow the query above has already terminated — lastProgress is populated.
p = q.lastProgress
if p is None:
    print("No batches completed yet.")
else:
    print(f"batchId:                {p['batchId']}")
    print(f"timestamp:              {p['timestamp']}")
    print(f"numInputRows:           {p['numInputRows']}")
    print(f"inputRowsPerSecond:     {p['inputRowsPerSecond']:.1f}")
    print(f"processedRowsPerSecond: {p['processedRowsPerSecond']:.1f}")
    print()
    print("Duration breakdown (ms):")
    for phase, ms in p.get("durationMs", {}).items():
        print(f"  {phase:<30} {ms}")
    print()
    if p.get("stateOperators"):
        print("State operators:")
        for op in p["stateOperators"]:
            print(f"  numRowsTotal={op.get('numRowsTotal')}  "
                  f"memoryUsedBytes={op.get('memoryUsedBytes')}")
    else:
        print("No stateful operators in this query.")

## 3. `query.recentProgress` — throughput over time

`recentProgress` is a list of the last 100 `lastProgress` dicts. Use it to spot trends: latency spikes, throughput drops, growing batch sizes.

In [ ]:
# Under availableNow, recentProgress holds one entry (the single batch). On a long-running
# stream this list would hold the last 100 batches and you'd plot rows/s over time.
recent = q.recentProgress
print(f"Batches in recentProgress: {len(recent)}")
print()
print(f"{'batchId':>8}  {'inputRows':>10}  {'rows/s':>8}  {'triggerMs':>10}")
print("-" * 44)
for p in recent:
    trigger_ms = p.get("durationMs", {}).get("triggerExecution", 0)
    print(f"{p['batchId']:>8}  {p['numInputRows']:>10}  "
          f"{p['inputRowsPerSecond']:>8.1f}  {trigger_ms:>10}")

## 4. `StreamingQueryListener` — event-driven monitoring

Polling `lastProgress` works for ad-hoc checks, but production systems need **push-based monitoring** — a callback that fires for every batch without sleeping in a loop.

`StreamingQueryListener` provides three callbacks:

| Callback | When it fires |
|---|---|
| `onQueryStarted(event)` | Immediately after `.start()` |
| `onQueryProgress(event)` | After every completed micro-batch |
| `onQueryTerminated(event)` | After the query stops (normally or with error) |

Attach it with `spark.streams.addListener(...)` **before** starting the query.

In [ ]:
from pyspark.sql.streaming import StreamingQueryListener

batch_metrics = []

class MetricsListener(StreamingQueryListener):
    def onQueryStarted(self, event):
        print(f"[listener] Query started: {event.id}")

    def onQueryProgress(self, event):
        prog = event.progress
        batch_metrics.append({
            "batchId":  prog.batchId,
            "inputRows": prog.numInputRows,
            "rows_per_s": prog.processedRowsPerSecond,
        })
        print(f"[listener] batch {prog.batchId}: "
              f"{prog.numInputRows} rows, "
              f"{prog.processedRowsPerSecond:.0f} rows/s")

    def onQueryTerminated(self, event):
        print(f"[listener] Query terminated: {event.id}  "
              f"exception={event.exception}")

listener = MetricsListener()
spark.streams.addListener(listener)

q2 = (events.select("event_id", "user_id", "amount")
      .writeStream.format("memory").queryName("listener_demo")
      .outputMode("append")
      .option("checkpointLocation", f"{CKPT_BASE}/listener_demo_{uuid.uuid4().hex[:8]}")
      .trigger(availableNow=True)
      .start())

q2.awaitTermination()

spark.streams.removeListener(listener)
print(f"\nCollected {len(batch_metrics)} batch metrics via listener.")

## 5. `spark.streams` — managing all active queries

In [ ]:
# Start two queries — under availableNow each runs once then stops, so we briefly see them active.
q_a = (events.select("event_id", "user_id")
       .writeStream.format("memory").queryName("query_a").outputMode("append")
       .option("checkpointLocation", f"{CKPT_BASE}/query_a_{uuid.uuid4().hex[:8]}")
       .trigger(availableNow=True).start())
q_b = (events.select("event_id", "amount")
       .writeStream.format("memory").queryName("query_b").outputMode("append")
       .option("checkpointLocation", f"{CKPT_BASE}/query_b_{uuid.uuid4().hex[:8]}")
       .trigger(availableNow=True).start())

print(f"Active queries while running: {len(spark.streams.active)}")
for aq in spark.streams.active:
    print(f"  name={aq.name:<15}  id={aq.id}  isActive={aq.isActive}")

q_a.awaitTermination()
q_b.awaitTermination()

print(f"\nAfter both terminated: {len(spark.streams.active)} active queries")

## 6. `awaitTermination` — clean shutdown and error propagation

`.awaitTermination()` blocks the current thread until the query stops (normally or on error). Use it to:
- Keep a notebook cell alive while the stream runs.
- Propagate exceptions from the streaming thread to the main thread.

| Call | Behaviour |
|---|---|
| `q.awaitTermination()` | Block forever — only returns if query stops |
| `q.awaitTermination(timeout)` | Block for at most `timeout` seconds; returns `True` if stopped, `False` if timed out |
| `spark.streams.awaitAnyTermination()` | Block until **any** active query terminates |
| `spark.streams.awaitAnyTermination(timeout)` | Same with timeout |

In [ ]:
# Pattern: run with availableNow and awaitTermination for a clean "batch" execution
spark.sql(
    "INSERT INTO population_metrics.default.avail_now_src VALUES "
    "(100, 0, 99, current_timestamp()),(101, 1, 55, current_timestamp())"
)

q_batch = (spark.readStream.format("delta")
    .table("population_metrics.default.avail_now_src")
    .writeStream.format("memory").queryName("await_demo")
    .outputMode("append")
    .option("checkpointLocation", f"{CKPT_BASE}/await_demo_{uuid.uuid4().hex[:8]}")
    .trigger(availableNow=True)
    .start())

finished = q_batch.awaitTermination(timeout=30)
print(f"Finished cleanly: {finished}")
print(f"Batches: {len(q_batch.recentProgress)}, "
      f"Rows: {sum(p['numInputRows'] for p in q_batch.recentProgress)}")

# Error propagation: if the query threw an exception, awaitTermination re-raises it
try:
    q_batch.awaitTermination()    # safe to call again after termination
except Exception as e:
    print(f"Query ended with error: {e}")

## 7. Common production pitfalls

| Symptom | Root cause | Fix |
|---|---|---|
| Query runs but `numInputRows` is always 0 | Source is empty or no new data since last checkpoint | Check source, verify checkpoint is reading the right offsets |
| Batch latency grows each cycle | Processing takes longer than trigger interval | Increase cluster size, reduce `maxFilesPerTrigger`, or tune shuffles |
| `StreamingQueryException: Connection refused` | Auth token expired (Databricks CLI) | Re-authenticate: `databricks auth login --profile <name>` |
| `OutOfMemoryError` in stateful operator | State growing without watermark | Add `.withWatermark(...)` before stateful ops |
| Duplicate rows in sink after restart | Sink is not idempotent | Use Delta MERGE or Delta's built-in exactly-once append |
| Checkpoint directory not found on restart | `/tmp` was cleared, or wrong path | Always use durable storage (Unity Catalog Volume, cloud storage) |
| Schema mismatch error on write | Column added/renamed in source | Enable `mergeSchema=true` or recreate checkpoint |
| `isActive=False` with no error | Query terminated silently | Check `q.exception()` and cluster logs |
| Slow first batch after idle period | Auth token refresh + connection re-establishment | Expected; reduce idle time or use keep-alive cluster |
| `Task failed: multiple times` | Data skew in shuffle | Add `OPTIMIZE ZORDER` on source, salt join keys |

## 8. Production checklist

Before deploying a streaming pipeline:

**Source**
- [ ] Checkpoint location is on durable storage (not `/tmp`)
- [ ] `startingVersion` / `startingTimestamp` set appropriately (not reading full history unexpectedly)
- [ ] `maxFilesPerTrigger` set to avoid giant first batches on large tables

**Transformations**
- [ ] All stateful operations have `.withWatermark(...)`
- [ ] State operator memory usage monitored via `lastProgress["stateOperators"]`

**Sink**
- [ ] Delta sink with checkpoint → exactly-once by default
- [ ] If using `foreachBatch`, writes are idempotent (MERGE or `batch_id` check)
- [ ] Schema enforcement verified — no silent schema drift

**Triggers**
- [ ] `processingTime` or `availableNow` chosen intentionally (no default continuous trigger burning compute)
- [ ] `availableNow` used for scheduled jobs to avoid idle cluster cost

**Monitoring**
- [ ] `StreamingQueryListener` or equivalent registered for operational alerting
- [ ] Alert on: `isActive=False`, `numInputRows=0` for N consecutive batches, state memory above threshold
- [ ] `q.exception()` checked in termination handler to distinguish clean stop from error

**Recovery**
- [ ] Runbook for re-auth token expiry (`databricks auth login --profile <name>`)
- [ ] Documented procedure for checkpoint deletion (when allowed and what it implies)
- [ ] Schema change process: which changes require a new checkpoint vs `mergeSchema=true`

## Takeaways

- **`query.status`** is a cheap, always-available health check. **`query.lastProgress`** is the rich per-batch diagnostic.
- **`recentProgress`** holds the last 100 batches — plot `processedRowsPerSecond` to spot throughput degradation.
- **`StreamingQueryListener`** is the production-grade monitoring hook — push metrics to Datadog, Prometheus, or a Delta log table from `onQueryProgress`.
- **`spark.streams.active`** lists all queries in the session — useful for cleanup and in multi-query notebooks.
- **`awaitTermination(timeout)`** is the correct way to keep a job alive; pair it with `q.exception()` for error propagation.
- The two most common silent failure modes: expired auth token (→ re-auth) and unbounded state growth (→ add watermark).
- Always use durable storage for checkpoints and always treat checkpoint deletion as a schema-breaking operation.